In [1]:
import sqlite3
import pandas as pd

def split_data_for_date(target_date, source_db='firm_data.db'):
    """
    Pulls data for a specific date from the main database, splits it into 
    two random batches, and saves them as separate SQLite databases.
    
    Args:
        target_date (str): Date in 'YYYY-MM-DD' format (e.g., '2020-12-31')
        source_db (str): The name of your main database
    """
    print(f"\n--- Splitting data for {target_date} ---")
    
    # 1. Pull data for the target date
    try:
        main_conn = sqlite3.connect(source_db)
        query = f"SELECT * FROM firms WHERE datadate = '{target_date}'"
        df = pd.read_sql_query(query, main_conn)
        main_conn.close()
    except Exception as e:
        print(f"❌ Error reading from {source_db}: {e}")
        return

    if df.empty:
        print(f"⚠️ No data found for date {target_date}. Please check the date format.")
        return

    print(f"Found {len(df)} firms for {target_date}.")

    # 2. Shuffle and split the data
    # Using random_state=42 ensures the same firms go into the same batches if rerun
    df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
    mid_point = len(df_shuffled) // 2
    
    batches = {
        "batch_1": df_shuffled.iloc[:mid_point],
        "batch_2": df_shuffled.iloc[mid_point:]
    }

    # 3. Save each batch to a new date-stamped database
    for batch_name, df_batch in batches.items():
        # Create database name like: firm_data_2020-12-31_batch_1.db
        db_filename = f"firm_data_{target_date}_{batch_name}.db"
        
        try:
            batch_conn = sqlite3.connect(db_filename)
            df_batch.to_sql('firms', batch_conn, if_exists='replace', index=False)
            
            # Recreate indexes for tool query performance
            cursor = batch_conn.cursor()
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_ticker ON firms(ticker)")
            cursor.execute("CREATE INDEX IF NOT EXISTS idx_date ON firms(datadate)")
            batch_conn.commit()
            batch_conn.close()
            
            print(f"✓ Saved {len(df_batch)} firms to '{db_filename}'")
        except Exception as e:
            print(f"❌ Error saving {db_filename}: {e}")


dates_to_split = ['2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31']

for date in dates_to_split:
    split_data_for_date(date)


--- Splitting data for 2020-12-31 ---
Found 497 firms for 2020-12-31.
✓ Saved 248 firms to 'firm_data_2020-12-31_batch_1.db'
✓ Saved 249 firms to 'firm_data_2020-12-31_batch_2.db'

--- Splitting data for 2021-12-31 ---
Found 497 firms for 2021-12-31.
✓ Saved 248 firms to 'firm_data_2021-12-31_batch_1.db'
✓ Saved 249 firms to 'firm_data_2021-12-31_batch_2.db'

--- Splitting data for 2022-12-31 ---
Found 499 firms for 2022-12-31.
✓ Saved 249 firms to 'firm_data_2022-12-31_batch_1.db'
✓ Saved 250 firms to 'firm_data_2022-12-31_batch_2.db'

--- Splitting data for 2023-12-31 ---
Found 500 firms for 2023-12-31.
✓ Saved 250 firms to 'firm_data_2023-12-31_batch_1.db'
✓ Saved 250 firms to 'firm_data_2023-12-31_batch_2.db'
